## Тест LLM-кандидатів — v2, з чіткою специфікацією

**Чому v2:** перший прогін (див. історію обговорення) показав хибні "перемоги" — модель
карали за формат/межі поля, яких їй ніхто не задав у промпті, і за нестабільність
OpenRouter (той самий запит на `temperature=0` іноді дав різну відповідь між прогонами).
Це виправлено тут:

1. **Явна специфікація формату й меж поля** — дата в `DD.MM.YYYY`, довільнотекстові поля
   у називному відмінку без зайвих слів, і явне правило для заглушок ("XXX", "_____" —
   вважати відсутністю значення, так само як null).
2. **N=3 повтори на кожен кейс** — щоб відрізнити реальну відмінність моделей від шуму
   маршрутизації OpenRouter. Звітуємо і середню точність, і чи прогони взагалі
   узгоджені між собою (`consistent_across_repeats`).

Задачі й дані (текст/еталон) ті самі, що в v1 — architecture-proposal.md, розділ 3:
екстракція шумних полів з обов'язковим null, резолюція терміну в довідник статусів,
вибір шаблону запиту + параметри. Скоринг — детермінований точний збіг, без LLM-судді.

Моделі: `qwen/qwen3-14b`, `mistralai/mistral-small-24b-instruct-2501` (обидві Apache 2.0,
підтверджено на HuggingFace).

In [1]:
import re, json, statistics
from openai import OpenAI

OPENROUTER_API_KEY = "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"
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)

MODELS = {
    "Qwen3-14B": "qwen/qwen3-14b",
    "Mistral-Small-24B-Instruct-2501": "mistralai/mistral-small-24b-instruct-2501",
}

N_REPEATS = 3  # перевіряємо стабільність відповіді, а не лише одну точку

def call_model(model_id: str, prompt: str) -> str:
    resp = client.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

def call_model_repeated(model_id: str, prompt: str, n: int = N_REPEATS) -> list:
    """n викликів (temperature=0) — перевіряє, чи OpenRouter справді детермінований тут."""
    return [call_model(model_id, prompt) for _ in range(n)]

def parse_json(text: str):
    """Витягує перший валідний JSON-об'єкт з відповіді моделі (код-блоки/зайвий текст навколо)."""
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.MULTILINE)
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

def normalize_value(v):
    """Легка нормалізація для порівняння: None-подібні рядки -> None, числові рядки -> int."""
    if v is None:
        return None
    if isinstance(v, str):
        v = v.strip().lower()
        if v in ("", "null", "невідомо", "не вказано", "n/a", "none"):
            return None
        if re.fullmatch(r"-?\d+", v):
            return int(v)
        return v
    return v

def dict_key(d):
    """Канонічний ключ для перевірки, чи всі повтори дали однаковий результат."""
    return json.dumps(d, sort_keys=True, ensure_ascii=False) if isinstance(d, dict) else repr(d)


### Задача А — екстракція шумних полів (з явним форматом і правилом для заглушок)

In [2]:
TASK_A_CASES = [
    {
        "label": "1.webp — заповнений рапорт (чистий скан)",
        "text": (
            "Прошу Вас надати мені частину щорічної основної відпустки за 2023 рік "
            "терміном на 10 діб з 01 січні 2023. Обов'язки помічника начальника штабу "
            "прошу покласти на офіцера штабу майора Коцюбу. Відпустку буду проводити за "
            "адресою: м. Вінниця, вул. Велика Бандерівська, б. 11 кв. 1155. "
            "Моб.телефон 067-891-23-45. У Збройних Силах України з 2014 року."
        ),
        "fields": ["vacation_type", "duration_days", "start_date", "end_date", "phone", "replacement_officer"],
        "golden": {
            "vacation_type": "щорічна основна відпустка",
            "duration_days": 10,
            "start_date": "01.01.2023",
            "end_date": None,
            "phone": "067-891-23-45",
            "replacement_officer": "майор Коцюба",
        },
    },
    {
        "label": "довідка ВЛК (синтетична, без приватних даних)",
        "text": (
            "ДОВІДКА військово-лікарської комісії. Проведено медичний огляд ВЛК "
            "клінічна лікарня м. Києва. Діагноз та постанова ВЛК про причинний зв'язок "
            "захворювання: Стан після мінно-вибухової травми (23.06.2023 р.). За наказом "
            "МОЗ від 04.07.2007 № 370 травма легка. Травма, ТАК, пов'язана з проходженням "
            "військової служби (довідка про обставини травми не надана). Потребує "
            "відпустки за станом здоров'я на 30 (тридцять) календарних днів."
        ),
        "fields": ["injury_date", "vacation_days", "service_related", "circumstances_certificate_provided", "rank"],
        "golden": {
            "injury_date": "23.06.2023",
            "vacation_days": 30,
            "service_related": True,
            "circumstances_certificate_provided": False,
            "rank": None,
        },
    },
    {
        "label": "public — порожній бланк (нічого не заповнено)",
        "text": (
            "Командиру В/ч _____. Рапорт. Прошу надати мені, _____ (звання, ПІБ), "
            "відпустку на _____ діб за сімейними обставинами. Відпустку буду проводити "
            "за адресою: _____. Тел.: _____."
        ),
        "fields": ["rank", "full_name", "duration_days", "address", "phone"],
        "golden": {
            "rank": None,
            "full_name": None,
            "duration_days": None,
            "address": None,
            "phone": None,
        },
    },
    {
        "label": "raport-optimized — знеособлений шаблон (XXX замість значень)",
        "text": (
            "Прошу вас надати мені частину щорічної основної відпустки терміном на "
            "10 (десять) діб із 01.10.2023 року. Відпустку буду проводити за адресою: "
            "країна XXX, місто XXX, вул. XXX. Телефон для оповіщення: XXX."
        ),
        "fields": ["duration_days", "start_date", "country", "city", "phone"],
        "golden": {
            "duration_days": 10,
            "start_date": "01.10.2023",
            "country": None,
            "city": None,
            "phone": None,
        },
    },
]

FORMAT_RULES = (
    "Правила формату (дотримуйся точно):\n"
    "1. Дати повертай СУВОРО у форматі DD.MM.YYYY (напр. «01 січня 2023 року» -> «01.01.2023»), "
    "незалежно від того, як вони записані в тексті.\n"
    "2. Довільнотекстові поля (звання, ім'я/прізвище, назви) повертай у називному відмінку "
    "(хто? що?) і БЕЗ додаткових слів навколо (напр. «майор Коцюба», а не «майора Коцюбу» "
    "чи «офіцер штабу майора Коцюби»).\n"
    "3. Якщо замість реального значення в тексті стоїть заглушка на кшталт «XXX», «_____» "
    "чи подібний символ-плейсхолдер — це те саме, що відсутність значення: постав null, "
    "а не копіюй заглушку як значення.\n"
    "4. Якщо поле не вказано в тексті явно — постав null. НЕ вигадуй і НЕ обчислюй значення, "
    "яких немає прямо в тексті (напр. не обчислюй дату закінчення за датою початку і "
    "тривалістю, якщо дата закінчення не вказана прямим текстом)."
)

# One-shot приклад — перевіряємо гіпотезу, що конкретний приклад очікуваного
# виводу (а не лише текстове правило) прибирає нестабільність між прогонами.
# Документ у прикладі синтетичний і НЕ перетинається з жодним із 4 тест-кейсів нижче.
FEW_SHOT_EXAMPLE = (
    "Приклад правильного застосування правил:\n\n"
    "Текст: \"Прошу надати відрядження терміном на 5 діб з 12 березня 2024 року. "
    "Обов'язки чергового по частині покласти на капітана Іваненка. "
    "Контактний номер: XXX.\"\n"
    "Поля для витягування: duration_days, start_date, responsible_officer, contact_phone\n"
    "Правильна відповідь: "
    '{"duration_days": 5, "start_date": "12.03.2024", '
    '"responsible_officer": "капітан Іваненко", "contact_phone": null}\n'
    "(зверни увагу: дата нормалізована в DD.MM.YYYY; звання+прізвище — у називному "
    "відмінку, без слів на кшталт \"черговий по частині\"; \"XXX\" розпізнано як "
    "заглушку і замінено на null.)"
)

def build_task_a_prompt(case: dict) -> str:
    fields_str = ", ".join(case["fields"])
    return (
        "Ти отримуєш уривок тексту українського військового документа. Витягни "
        f"вказані поля у форматі JSON: {fields_str}.\n\n"
        f"{FORMAT_RULES}\n\n"
        f"{FEW_SHOT_EXAMPLE}\n\n"
        f"Текст документа:\n\"\"\"\n{case['text']}\n\"\"\"\n\n"
        "Поверни ЛИШЕ JSON-об'єкт з цими полями, без пояснень і без markdown-обгортки."
    )

def score_task_a(golden: dict, predicted):
    if not isinstance(predicted, dict):
        return {"valid_json": False, "field_scores": {}, "accuracy": 0.0}
    field_scores = {}
    for field, gold_val in golden.items():
        pred_val = normalize_value(predicted.get(field))
        gold_norm = normalize_value(gold_val)
        field_scores[field] = (pred_val == gold_norm)
    accuracy = sum(field_scores.values()) / len(field_scores)
    return {"valid_json": True, "field_scores": field_scores, "accuracy": accuracy}

task_a_results = []
for model_name, model_id in MODELS.items():
    for case in TASK_A_CASES:
        prompt = build_task_a_prompt(case)
        raws = call_model_repeated(model_id, prompt)
        parsed_list = [parse_json(r) for r in raws]
        scores = [score_task_a(case["golden"], p) for p in parsed_list]
        accuracies = [s["accuracy"] for s in scores]
        consistent = len({dict_key(p) for p in parsed_list}) == 1
        result = {
            "model": model_name, "case": case["label"],
            "mean_accuracy": statistics.mean(accuracies),
            "min_accuracy": min(accuracies), "max_accuracy": max(accuracies),
            "consistent_across_repeats": consistent,
            "parsed_list": parsed_list, "field_scores_list": [s["field_scores"] for s in scores],
        }
        task_a_results.append(result)
        stability = "стабільно" if consistent else "НЕСТАБІЛЬНО між прогонами"
        print(f"[{model_name}] {case['label']}: точність={result['mean_accuracy']:.0%} "
              f"(min={result['min_accuracy']:.0%}, max={result['max_accuracy']:.0%}) — {stability}")


[Qwen3-14B] 1.webp — заповнений рапорт (чистий скан): точність=89% (min=83%, max=100%) — НЕСТАБІЛЬНО між прогонами


[Qwen3-14B] довідка ВЛК (синтетична, без приватних даних): точність=73% (min=60%, max=100%) — НЕСТАБІЛЬНО між прогонами


[Qwen3-14B] public — порожній бланк (нічого не заповнено): точність=100% (min=100%, max=100%) — стабільно


[Qwen3-14B] raport-optimized — знеособлений шаблон (XXX замість значень): точність=100% (min=100%, max=100%) — стабільно


[Mistral-Small-24B-Instruct-2501] 1.webp — заповнений рапорт (чистий скан): точність=83% (min=83%, max=83%) — НЕСТАБІЛЬНО між прогонами


[Mistral-Small-24B-Instruct-2501] довідка ВЛК (синтетична, без приватних даних): точність=100% (min=100%, max=100%) — стабільно


[Mistral-Small-24B-Instruct-2501] public — порожній бланк (нічого не заповнено): точність=100% (min=100%, max=100%) — стабільно


[Mistral-Small-24B-Instruct-2501] raport-optimized — знеособлений шаблон (XXX замість значень): точність=100% (min=100%, max=100%) — стабільно


Деталі по полях (де саме і як часто модель помилялась за N_REPEATS прогони):

In [3]:
for r in task_a_results:
    # частота помилки на кожне поле серед усіх повторів
    field_names = r["field_scores_list"][0].keys() if r["field_scores_list"] else []
    error_rates = {
        f: sum(1 for fs in r["field_scores_list"] if not fs.get(f, False)) / len(r["field_scores_list"])
        for f in field_names
    }
    wrong_fields = {f: rate for f, rate in error_rates.items() if rate > 0}
    if wrong_fields:
        print(f"[{r['model']}] {r['case']}: частота помилки по полях {wrong_fields}")
        print(f"  приклад відповіді: {r['parsed_list'][0]}")
        print()


[Qwen3-14B] 1.webp — заповнений рапорт (чистий скан): частота помилки по полях {'vacation_type': 0.6666666666666666}
  приклад відповіді: {'vacation_type': 'щорічної основної відпустки', 'duration_days': 10, 'start_date': '01.01.2023', 'end_date': None, 'phone': '067-891-23-45', 'replacement_officer': 'майор Коцюба'}

[Qwen3-14B] довідка ВЛК (синтетична, без приватних даних): частота помилки по полях {'service_related': 0.6666666666666666, 'circumstances_certificate_provided': 0.6666666666666666}
  приклад відповіді: {'injury_date': '23.06.2023', 'vacation_days': 30, 'service_related': True, 'circumstances_certificate_provided': False, 'rank': None}

[Mistral-Small-24B-Instruct-2501] 1.webp — заповнений рапорт (чистий скан): частота помилки по полях {'vacation_type': 1.0}
  приклад відповіді: {'vacation_type': 'частину щорічної основної відпустки за 2023 рік', 'duration_days': 10, 'start_date': '01.01.2023', 'end_date': None, 'phone': '067-891-23-45', 'replacement_officer': 'майор Коцю

### Задача Б — резолюція терміну в контрольований довідник статусів

In [4]:
STATUS_VOCAB = ["vacation_active", "deployment_active", "discharged_from_service", "hospital_treatment", "unknown_term"]

# Прибрано кейс "на реабілітації після поранення" — визнаний невалідним тестом
# (межа словника для цього терміну не визначена нами односторонньо, це питання
# для профільної команди/військових, не для тесту моделі).
TASK_B_CASES = [
    {"term": "у відпустці", "golden": "vacation_active"},
    {"term": "перебуває у щорічній відпустці", "golden": "vacation_active"},
    {"term": "у відрядженні", "golden": "deployment_active"},
    {"term": "звільнений з військової служби", "golden": "discharged_from_service"},
    {"term": "проходить лікування у шпиталі", "golden": "hospital_treatment"},
    {"term": "у самоволці", "golden": "unknown_term"},
]

def build_task_b_prompt(term: str) -> str:
    vocab_str = ", ".join(STATUS_VOCAB)
    return (
        f"Тобі дано контрольований довідник статусів: {vocab_str}.\n"
        "Визнач, якому статусу з цього довідника відповідає наведений термін. "
        "Якщо термін НЕ відповідає жодному статусу з довідника — поверни "
        "\"unknown_term\". Не вигадуй статус поза довідником.\n\n"
        f"Термін: \"{term}\"\n\n"
        "Поверни ЛИШЕ JSON у форматі {\"status\": \"...\"}, без пояснень."
    )

task_b_results = []
for model_name, model_id in MODELS.items():
    for case in TASK_B_CASES:
        prompt = build_task_b_prompt(case["term"])
        raws = call_model_repeated(model_id, prompt)
        predicted_list = [
            (parse_json(r) or {}).get("status") if isinstance(parse_json(r), dict) else None
            for r in raws
        ]
        correct_rate = sum(1 for p in predicted_list if p == case["golden"]) / len(predicted_list)
        consistent = len(set(predicted_list)) == 1
        task_b_results.append({
            "model": model_name, "term": case["term"], "golden": case["golden"],
            "predicted_list": predicted_list, "correct_rate": correct_rate,
            "consistent_across_repeats": consistent,
        })
        stability = "стабільно" if consistent else "НЕСТАБІЛЬНО"
        print(f"[{model_name}] \"{case['term']}\" -> {predicted_list} "
              f"(очікувалось {case['golden']}, correct_rate={correct_rate:.0%}) — {stability}")


[Qwen3-14B] "у відпустці" -> ['vacation_active', 'vacation_active', 'vacation_active'] (очікувалось vacation_active, correct_rate=100%) — стабільно


[Qwen3-14B] "перебуває у щорічній відпустці" -> ['vacation_active', 'vacation_active', 'vacation_active'] (очікувалось vacation_active, correct_rate=100%) — стабільно


[Qwen3-14B] "у відрядженні" -> ['deployment_active', 'deployment_active', 'deployment_active'] (очікувалось deployment_active, correct_rate=100%) — стабільно


[Qwen3-14B] "звільнений з військової служби" -> ['discharged_from_service', 'discharged_from_service', 'discharged_from_service'] (очікувалось discharged_from_service, correct_rate=100%) — стабільно


[Qwen3-14B] "проходить лікування у шпиталі" -> ['hospital_treatment', 'hospital_treatment', 'hospital_treatment'] (очікувалось hospital_treatment, correct_rate=100%) — стабільно


...

[Mistral-Small-24B-Instruct-2501] "перебуває у щорічній відпустці" -> ['vacation_active', 'vacation_active', 'vacation_active'] (очікувалось vacation_active, correct_rate=100%) — стабільно


[Mistral-Small-24B-Instruct-2501] "у відрядженні" -> ['deployment_active', 'deployment_active', 'deployment_active'] (очікувалось deployment_active, correct_rate=100%) — стабільно


[Mistral-Small-24B-Instruct-2501] "звільнений з військової служби" -> ['discharged_from_service', 'discharged_from_service', 'discharged_from_service'] (очікувалось discharged_from_service, correct_rate=100%) — стабільно


[Mistral-Small-24B-Instruct-2501] "проходить лікування у шпиталі" -> ['hospital_treatment', 'hospital_treatment', 'hospital_treatment'] (очікувалось hospital_treatment, correct_rate=100%) — стабільно


[Mistral-Small-24B-Instruct-2501] "у самоволці" -> ['unknown_term', 'unknown_term', 'unknown_term'] (очікувалось unknown_term, correct_rate=100%) — стабільно


### Задача В — вибір шаблону агрегованого запиту + параметри

In [5]:
QUERY_TEMPLATES = """
- COUNT_BY_STATUS(status) \u2014 порахувати людей за статусом
- LIST_BY_STATUS(status) \u2014 перелічити людей за статусом
- COUNT_BY_STATUS_UNIT(status, unit) \u2014 порахувати за статусом у конкретному підрозділі
- COUNT_BY_STATUS_DATE_RANGE(status, date_from, date_to) \u2014 порахувати за статусом у діапазоні дат
- UNSUPPORTED \u2014 жоден шаблон не підходить
""".strip()

TASK_C_CASES = [
    {
        "question": "Скільки людей зараз у відпустці?",
        "golden": {"template": "COUNT_BY_STATUS", "params": {"status": "vacation_active"}},
    },
    {
        "question": "Хто зараз у відрядженні?",
        "golden": {"template": "LIST_BY_STATUS", "params": {"status": "deployment_active"}},
    },
    {
        "question": "Скільки людей у відпустці в 3 батальйоні?",
        "golden": {"template": "COUNT_BY_STATUS_UNIT", "params": {"status": "vacation_active", "unit": "3 батальйон"}},
    },
    {
        "question": "Скільки було звільнено з 01.06.2023 по 31.08.2023?",
        "golden": {
            "template": "COUNT_BY_STATUS_DATE_RANGE",
            "params": {"status": "discharged_from_service", "date_from": "01.06.2023", "date_to": "31.08.2023"},
        },
    },
    {
        "question": "Яка погода в Києві сьогодні?",
        "golden": {"template": "UNSUPPORTED", "params": {}},
    },
]

def build_task_c_prompt(question: str) -> str:
    return (
        f"Тобі дано фіксований список шаблонів запитів до бази даних:\n{QUERY_TEMPLATES}\n\n"
        "Обери, який шаблон найкраще відповідає запитанню користувача, і заповни "
        "параметри з тексту запитання. Використовуй ЛИШЕ статуси з цього довідника "
        f"для поля status: {', '.join(STATUS_VOCAB[:-1])}. Параметр unit повертай так, "
        "як він природно називається в питанні (напр. «3 батальйон», а не лише число). "
        "Якщо жоден шаблон не підходить — поверни {\"template\": \"UNSUPPORTED\", \"params\": {}}.\n\n"
        f"Запитання користувача: \"{question}\"\n\n"
        "Поверни ЛИШЕ JSON з полями \"template\" та \"params\" (об'єкт параметрів), без пояснень."
    )

def score_task_c(golden: dict, predicted):
    if not isinstance(predicted, dict):
        return False
    if predicted.get("template") != golden["template"]:
        return False
    pred_params = {k: normalize_value(v) for k, v in (predicted.get("params") or {}).items()}
    gold_params = {k: normalize_value(v) for k, v in golden["params"].items()}
    return pred_params == gold_params

task_c_results = []
for model_name, model_id in MODELS.items():
    for case in TASK_C_CASES:
        prompt = build_task_c_prompt(case["question"])
        raws = call_model_repeated(model_id, prompt)
        parsed_list = [parse_json(r) for r in raws]
        correct_list = [score_task_c(case["golden"], p) for p in parsed_list]
        correct_rate = sum(correct_list) / len(correct_list)
        consistent = len({dict_key(p) for p in parsed_list}) == 1
        task_c_results.append({
            "model": model_name, "question": case["question"], "golden": case["golden"],
            "parsed_list": parsed_list, "correct_rate": correct_rate,
            "consistent_across_repeats": consistent,
        })
        stability = "стабільно" if consistent else "НЕСТАБІЛЬНО"
        print(f"[{model_name}] \"{case['question']}\" -> {parsed_list} "
              f"(correct_rate={correct_rate:.0%}) — {stability}")


[Qwen3-14B] "Скільки людей зараз у відпустці?" -> [{'template': 'COUNT_BY_STATUS', 'params': {'status': 'vacation_active'}}, {'template': 'COUNT_BY_STATUS', 'params': {'status': 'vacation_active'}}, {'template': 'COUNT_BY_STATUS', 'params': {'status': 'vacation_active'}}] (correct_rate=100%) — стабільно


[Qwen3-14B] "Хто зараз у відрядженні?" -> [{'template': 'LIST_BY_STATUS', 'params': {'status': 'deployment_active'}}, {'template': 'LIST_BY_STATUS', 'params': {'status': 'deployment_active'}}, {'template': 'LIST_BY_STATUS', 'params': {'status': 'deployment_active'}}] (correct_rate=100%) — стабільно


[Qwen3-14B] "Скільки людей у відпустці в 3 батальйоні?" -> [{'template': 'COUNT_BY_STATUS_UNIT', 'params': {'status': 'vacation_active', 'unit': '3 батальйон'}}, {'template': 'COUNT_BY_STATUS_UNIT', 'params': {'status': 'vacation_active', 'unit': '3 батальйон'}}, {'template': 'COUNT_BY_STATUS_UNIT', 'params': {'status': 'vacation_active', 'unit': '3 батальйон'}}] (correct_rate=100%) — стабільно


[Qwen3-14B] "Скільки було звільнено з 01.06.2023 по 31.08.2023?" -> [{'template': 'COUNT_BY_STATUS_DATE_RANGE', 'params': {'status': 'discharged_from_service', 'date_from': '01.06.2023', 'date_to': '31.08.2023'}}, {'template': 'COUNT_BY_STATUS_DATE_RANGE', 'params': {'status': 'discharged_from_service', 'date_from': '01.06.2023', 'date_to': '31.08.2023'}}, {'template': 'COUNT_BY_STATUS_DATE_RANGE', 'params': {'status': 'discharged_from_service', 'date_from': '01.06.2023', 'date_to': '31.08.2023'}}] (correct_rate=100%) — стабільно


[Qwen3-14B] "Яка погода в Києві сьогодні?" -> [{'template': 'UNSUPPORTED', 'params': {}}, {'template': 'UNSUPPORTED', 'params': {}}, {'template': 'UNSUPPORTED', 'params': {}}] (correct_rate=100%) — стабільно


[Mistral-Small-24B-Instruct-2501] "Скільки людей зараз у відпустці?" -> [{'template': 'COUNT_BY_STATUS', 'params': {'status': 'vacation_active'}}, {'template': 'COUNT_BY_STATUS', 'params': {'status': 'vacation_active'}}, {'template': 'COUNT_BY_STATUS', 'params': {'status': 'vacation_active'}}] (correct_rate=100%) — стабільно


[Mistral-Small-24B-Instruct-2501] "Хто зараз у відрядженні?" -> [{'template': 'LIST_BY_STATUS', 'params': {'status': 'deployment_active'}}, {'template': 'LIST_BY_STATUS', 'params': {'status': 'deployment_active'}}, {'template': 'LIST_BY_STATUS', 'params': {'status': 'deployment_active'}}] (correct_rate=100%) — стабільно


[Mistral-Small-24B-Instruct-2501] "Скільки людей у відпустці в 3 батальйоні?" -> [{'template': 'COUNT_BY_STATUS_UNIT', 'params': {'status': 'vacation_active', 'unit': '3 батальйон'}}, {'template': 'COUNT_BY_STATUS_UNIT', 'params': {'status': 'vacation_active', 'unit': '3 батальйон'}}, {'template': 'COUNT_BY_STATUS_UNIT', 'params': {'status': 'vacation_active', 'unit': '3 батальйон'}}] (correct_rate=100%) — стабільно


[Mistral-Small-24B-Instruct-2501] "Скільки було звільнено з 01.06.2023 по 31.08.2023?" -> [{'template': 'COUNT_BY_STATUS_DATE_RANGE', 'params': {'status': 'discharged_from_service', 'date_from': '01.06.2023', 'date_to': '31.08.2023'}}, {'template': 'COUNT_BY_STATUS_DATE_RANGE', 'params': {'status': 'discharged_from_service', 'date_from': '01.06.2023', 'date_to': '31.08.2023'}}, {'template': 'COUNT_BY_STATUS_DATE_RANGE', 'params': {'status': 'discharged_from_service', 'date_from': '01.06.2023', 'date_to': '31.08.2023'}}] (correct_rate=100%) — стабільно


[Mistral-Small-24B-Instruct-2501] "Яка погода в Києві сьогодні?" -> [{'template': 'UNSUPPORTED', 'params': {}}, {'template': 'UNSUPPORTED', 'params': {}}, {'template': 'UNSUPPORTED', 'params': {}}] (correct_rate=100%) — стабільно


### Підсумок

In [6]:
print("=" * 70)
print("ПІДСУМОК (середнє за N=3 прогони; consistency = чи прогони збігались)")
print("=" * 70)
for model_name in MODELS:
    a = [r for r in task_a_results if r["model"] == model_name]
    b = [r for r in task_b_results if r["model"] == model_name]
    c = [r for r in task_c_results if r["model"] == model_name]
    a_acc = statistics.mean(r["mean_accuracy"] for r in a)
    a_consistent = sum(r["consistent_across_repeats"] for r in a)
    b_acc = statistics.mean(r["correct_rate"] for r in b)
    b_consistent = sum(r["consistent_across_repeats"] for r in b)
    c_acc = statistics.mean(r["correct_rate"] for r in c)
    c_consistent = sum(r["consistent_across_repeats"] for r in c)
    print(f"\n{model_name}:")
    print(f"  Задача А: точність={a_acc:.0%}, стабільно на {a_consistent}/{len(a)} кейсів")
    print(f"  Задача Б: точність={b_acc:.0%}, стабільно на {b_consistent}/{len(b)} кейсів")
    print(f"  Задача В: точність={c_acc:.0%}, стабільно на {c_consistent}/{len(c)} кейсів")


ПІДСУМОК (середнє за N=3 прогони; consistency = чи прогони збігались)

Qwen3-14B:
  Задача А: точність=91%, стабільно на 2/4 кейсів
  Задача Б: точність=100%, стабільно на 6/6 кейсів
  Задача В: точність=100%, стабільно на 5/5 кейсів

Mistral-Small-24B-Instruct-2501:
  Задача А: точність=96%, стабільно на 3/4 кейсів
  Задача Б: точність=100%, стабільно на 6/6 кейсів
  Задача В: точність=100%, стабільно на 5/5 кейсів
